In [1]:
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [2]:
from config import (
    BEST_HUMAN_CUES_PATH,
    SEED_EXAMPLES_PATH,
    CATEGORY_PROMPT_CANDIDATES_PATH,
    CATEGORY_INITIAL_RESULTS_PATH,
)
from prompt_utils import sample_demo_examples, format_demo_block
from llm_utils import generate_therapy_cue

Using local model path: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\models\Qwen2.5-1.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
best_human_df = pd.read_csv(BEST_HUMAN_CUES_PATH)
seed_df = pd.read_csv(SEED_EXAMPLES_PATH)
category_prompt_df = pd.read_csv(CATEGORY_PROMPT_CANDIDATES_PATH)

print("Prompt rows:", len(category_prompt_df))
print("Words:", len(best_human_df))

Prompt rows: 72
Words: 130


In [4]:
rows = []

for _, p_row in tqdm(category_prompt_df.iterrows(), total=len(category_prompt_df), desc="Category prompt candidates"):
    subcategory = p_row["subcategory"]
    eval_subset = best_human_df[best_human_df["subcategory"] == subcategory].copy()

    for _, ex in tqdm(eval_subset.iterrows(), total=len(eval_subset), desc=subcategory, leave=False):
        demo_examples = sample_demo_examples(
            seed_df=seed_df,
            subcategory=subcategory,
            n_examples=3,
            random_state=42,
        )
        demo_block = format_demo_block(demo_examples)

        generated_cue = generate_therapy_cue(
            instruction=p_row["prompt_text"],
            target_word=ex["target_word"],
            pos=ex["pos"],
            subcategory=subcategory,
            demo_block=demo_block,
        )

        rows.append({
            "ape_round": 0,
            "parent_prompt_id": "",
            "subcategory": subcategory,
            "prompt_id": p_row["prompt_id"],
            "prompt_family": p_row["prompt_family"],
            "prompt_variant": p_row["prompt_variant"],
            "prompt_text": p_row["prompt_text"],
            "target_word": ex["target_word"],
            "pos": ex["pos"],
            "reference_cue": ex["cue_text"],
            "generated_cue": generated_cue,
        })

category_initial_results_df = pd.DataFrame(rows)
category_initial_results_df.head()

Category prompt candidates: 100%|██████████| 72/72 [09:44<00:00,  8.12s/it]


,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_variant,prompt_text,target_word,pos,reference_cue,generated_cue
0,0,,交通場所,交通場所_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,
1,0,,交通場所,交通場所_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,
2,0,,交通場所,交通場所_p03,category_meta,v3_life_context,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,
3,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,需要刷卡進站
4,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,摩托車,名詞,需要加汽油或電力才能運行,需要開闢道路才能行駛


In [5]:
print("category_initial_results_df shape:", category_initial_results_df.shape)
display(category_initial_results_df.head(10))

category_initial_results_df shape: (390, 11)


,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_variant,prompt_text,target_word,pos,reference_cue,generated_cue
0,0,,交通場所,交通場所_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,
1,0,,交通場所,交通場所_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,
2,0,,交通場所,交通場所_p03,category_meta,v3_life_context,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,
3,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,需要刷卡進站
4,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,摩托車,名詞,需要加汽油或電力才能運行,需要開闢道路才能行駛
5,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,汽車,名詞,需要加油才能行駛,你需要在等人時看手機
6,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,火車,名詞,"需要購票搭乘,有分自由座和對號座",需要購票才能登車
7,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,自行車,名詞,運動或代步皆適合,騎行方便，速度快，適合短途出行
8,0,,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,高鐵,名詞,"透過電力驅動,行駛於專用軌道上",需要刷卡才能進站
9,0,,交通工具,交通工具_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,上車時要留意刷卡機，按正確順序購買車票


In [6]:
category_initial_results_df.to_csv(
    CATEGORY_INITIAL_RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)
print("Saved:", CATEGORY_INITIAL_RESULTS_PATH)

Saved: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\interim\category_initial_generation_results.csv


In [7]:
check_df = pd.read_csv(CATEGORY_INITIAL_RESULTS_PATH)
print(check_df.shape)
display(check_df.head(10))

(390, 11)


,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_variant,prompt_text,target_word,pos,reference_cue,generated_cue
0,0,NaN,交通場所,交通場所_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,NaN
1,0,NaN,交通場所,交通場所_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,NaN
2,0,NaN,交通場所,交通場所_p03,category_meta,v3_life_context,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通場所詞彙，並...,機場,名詞,內部有免稅商店和餐廳,NaN
3,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,需要刷卡進站
4,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,摩托車,名詞,需要加汽油或電力才能運行,需要開闢道路才能行駛
5,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,汽車,名詞,需要加油才能行駛,你需要在等人時看手機
6,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,火車,名詞,"需要購票搭乘,有分自由座和對號座",需要購票才能登車
7,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,自行車,名詞,運動或代步皆適合,騎行方便，速度快，適合短途出行
8,0,NaN,交通工具,交通工具_p01,category_meta,v1_meta_base,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,高鐵,名詞,"透過電力驅動,行駛於專用軌道上",需要刷卡才能進站
9,0,NaN,交通工具,交通工具_p02,category_meta,v2_discriminative,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,公車,名詞,票價便宜,上車時要留意刷卡機，按正確順序購買車票
